# YT Archive engine

Press **Run** on the cell below, wait about a minute, then go back to the YT Archive page and paste the code it prints.
Leave the cell running while you use the page.

In [ ]:
import json, logging, re, secrets, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path

PAGE_URL = "https://miliketstudio.github.io/MiliketStudio/yt-archive.html"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "yt-dlp[default]", "img2pdf", "python-pptx", "flask", "flask-cors"], check=True)
CF = Path("/usr/local/bin/cloudflared")
if not CF.exists():
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   f"cloudflared-linux-amd64 -O {CF} && chmod +x {CF}", shell=True, check=True)

import numpy as np
from flask import Flask, abort, jsonify, request, send_file, send_from_directory
from flask_cors import CORS

ROOT, WORK = Path("/content/archive"), Path("/content/work")
ROOT.mkdir(exist_ok=True)
KEY = globals().get("KEY") or secrets.token_urlsafe(9)   # survives a re-run of this cell
JOBS = globals().get("JOBS", [])
RUN = globals().get("RUN") or threading.Lock()            # one job at a time


# ---------------- pipeline (same logic as before, now with progress) ----------------
def safe_name(s, n=40):
    s = re.sub(r'[\\/:*?"<>|\r\n\t]+', " ", s or "").strip()
    return s[:n].strip() or "video"


def ytcmd():
    c = [sys.executable, "-m", "yt_dlp"]
    return c + ["--js-runtimes", "node"] if shutil.which("node") else c


def list_entries(url):
    if "playlist?list=" not in url:
        return [(None, url)]
    out = subprocess.run(ytcmd() + ["--flat-playlist", "-J", url], check=True,
                         stdout=subprocess.PIPE, text=True, encoding="utf-8").stdout
    items = []
    for e in json.loads(out).get("entries") or []:
        if e:
            link = e.get("url") or ""
            if not link.startswith("http"):
                link = "https://www.youtube.com/watch?v=" + e["id"]
            items.append((e.get("title"), link))
    return items


def download(url, height, it):
    fmt = (f"bv*[vcodec^=avc1][height<={height}]+ba[acodec^=mp4a]/bv*[height<={height}]+ba"
           f"/b[height<={height}]/b")
    cmd = ytcmd() + ["--no-playlist", "--newline", "-f", fmt, "--merge-output-format", "mp4",
                     "-o", str(WORK / "video.%(ext)s"), url]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8")
    tail = []
    for line in p.stdout:
        tail = (tail + [line.strip()])[-6:]
        m = re.search(r"\[download\]\s+([\d.]+)%", line)
        if m:
            it["pct"] = float(m.group(1))
    if p.wait():
        raise RuntimeError(" ".join(tail)[-300:])


def find_changes(video, threshold, max_frames):
    w, h = 160, 90
    size = w * h
    proc = subprocess.Popen(["ffmpeg", "-v", "error", "-i", str(video), "-vf",
                             f"fps=1,scale={w}:{h},format=gray", "-f", "rawvideo", "-"],
                            stdout=subprocess.PIPE)
    kept, last_kept, prev, pending, t = [], None, None, 0, 0
    while True:
        buf = proc.stdout.read(size)
        if len(buf) < size:
            break
        cur = np.frombuffer(buf, dtype=np.uint8).astype(np.int16)
        if last_kept is None:
            kept.append(t)
            last_kept = cur
        else:
            changed = np.abs(cur - last_kept).mean() > threshold
            settled = np.abs(cur - prev).mean() < 1.0
            pending = pending + 1 if changed else 0
            if changed and (settled or pending >= 8):
                kept.append(t)
                last_kept = cur
                pending = 0
        prev = cur
        t += 1
    proc.wait()
    if len(kept) > max_frames:
        step = len(kept) / max_frames
        kept = [kept[int(i * step)] for i in range(max_frames)]
    return kept


def extract(video, times, frames_dir, it):
    frames_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for i, t in enumerate(times, 1):
        f = frames_dir / f"{i:04d}.jpg"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(t), "-i", str(video),
                        "-frames:v", "1", "-vf", "scale='min(1600,iw)':-2", "-q:v", "3", str(f)],
                       check=True)
        if f.exists():
            files.append(f)
        it["pct"] = 100.0 * i / len(times)
    return files


def build_pdf(files, out):
    import img2pdf
    out.write_bytes(img2pdf.convert([str(f) for f in files]))


def build_pptx(files, times, out):
    from PIL import Image
    from pptx import Presentation
    from pptx.util import Inches
    w, h = Image.open(files[0]).size
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = int(prs.slide_width * h / w)
    for f, t in zip(files, times):
        s = prs.slides.add_slide(prs.slide_layouts[6])
        s.shapes.add_picture(str(f), 0, 0, width=prs.slide_width, height=prs.slide_height)
        s.notes_slide.notes_text_frame.text = time.strftime("%H:%M:%S", time.gmtime(t))
    prs.save(str(out))


def build_zip(files, times, out):
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for f in files:
            z.write(f, f.name)
        z.writestr("frames.json", json.dumps([{"file": f.name, "t": t} for f, t in zip(files, times)]))


def process(it, o):
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir(parents=True)
    ch = o["outputs"]
    it.update(state="downloading", pct=0)
    if not it["title"]:
        r = subprocess.run(ytcmd() + ["--no-playlist", "--print", "title", it["url"]], check=True,
                           stdout=subprocess.PIPE, text=True, encoding="utf-8").stdout.strip()
        it["title"] = r.splitlines()[0] if r else it["url"]
    download(it["url"], o["quality"], it)
    video = WORK / "video.mp4"
    nums = [int(d.name[:3]) for d in ROOT.iterdir() if d.is_dir() and d.name[:3].isdigit()]
    out = ROOT / f"{max(nums, default=0) + 1:03d} - {safe_name(it['title'])}"
    out.mkdir()
    slides = 0
    if any(ch.get(k) for k in ("browse", "pdf", "pptx", "frames")):
        it.update(state="analysing", pct=None)
        times = find_changes(video, o["sensitivity"], o["max_frames"])
        it.update(state="extracting", pct=0)
        files = extract(video, times, out / "frames", it)
        times = times[:len(files)]
        if not files:
            raise RuntimeError("no frames could be extracted")
        slides = len(files)
        (out / "frames.json").write_text(json.dumps([{"file": f.name, "t": t} for f, t in zip(files, times)]))
        it.update(state="building", pct=None)
        if ch.get("pdf"):
            build_pdf(files, out / "slides.pdf")
        if ch.get("pptx"):
            build_pptx(files, times, out / "slides.pptx")
        if ch.get("frames"):
            build_zip(files, times, out / "frames.zip")
        if not ch.get("browse"):
            shutil.rmtree(out / "frames")
            (out / "frames.json").unlink()
    if ch.get("video"):
        shutil.copy(video, out / "video.mp4")
    names = [p.name for p in out.iterdir() if p.suffix in (".pdf", ".pptx", ".zip", ".mp4")]
    (out / "meta.json").write_text(json.dumps({"title": it["title"], "url": it["url"], "slides": slides,
                                               "browse": bool(ch.get("browse")), "files": sorted(names)}),
                                   encoding="utf-8")
    it.update(state="done", pct=100)


def run_job(job):
    with RUN:
        job["status"] = "listing"
        items = []
        for u in job["urls"]:
            try:
                entries = list_entries(u)
            except Exception as e:
                entries = []
                job["items"].append({"title": None, "url": u, "state": "failed", "pct": None,
                                     "msg": "could not read this link: " + str(e)[-200:]})
            for title, link in entries:
                it = {"title": title, "url": link, "state": "waiting", "pct": None, "msg": ""}
                items.append(it)
                job["items"].append(it)
        job["status"] = "running"
        for it in items:
            try:
                process(it, job["opts"])
            except Exception as e:
                it.update(state="failed", pct=None, msg=str(e)[-300:])
        job["status"] = "done"


# ---------------- API ----------------
app = Flask(__name__)
CORS(app)


@app.before_request
def guard():
    if request.method != "OPTIONS" and request.args.get("k") != KEY:
        abort(403)


def vdir(vid):
    d = ROOT / vid
    if "/" in vid or ".." in vid or not d.is_dir():
        abort(404)
    return d


@app.route("/api/ping")
def ping():
    return jsonify(ok=True)


@app.route("/api/jobs", methods=["GET", "POST"])
def jobs():
    if request.method == "GET":
        return jsonify(JOBS[-5:][::-1])
    b = request.get_json(force=True)
    urls = [u.strip() for u in b.get("urls", []) if u.strip()]
    if not urls:
        abort(400)
    clamp = lambda v, lo, hi: max(lo, min(hi, v))
    opts = {"outputs": {k: bool(v) for k, v in (b.get("outputs") or {}).items()},
            "sensitivity": clamp(float(b.get("sensitivity", 3)), 0.5, 20),
            "max_frames": clamp(int(b.get("max_frames", 300)), 5, 1000),
            "quality": clamp(int(b.get("quality", 1080)), 360, 2160)}
    job = {"id": secrets.token_hex(4), "status": "queued", "urls": urls, "items": [], "opts": opts}
    JOBS.append(job)
    threading.Thread(target=run_job, args=(job,), daemon=True).start()
    return jsonify(id=job["id"])


@app.route("/api/library")
def library():
    res = []
    for d in sorted(ROOT.iterdir()):
        if d.is_dir() and (d / "meta.json").exists():
            x = json.loads((d / "meta.json").read_text(encoding="utf-8"))
            x["id"] = d.name
            res.append(x)
    return jsonify(res)


@app.route("/api/library/<vid>/slides")
def slides(vid):
    return send_from_directory(vdir(vid), "frames.json")


@app.route("/api/library/<vid>/frames/<name>")
def frame(vid, name):
    return send_from_directory(vdir(vid) / "frames", name)


@app.route("/api/library/<vid>/files/<name>")
def getfile(vid, name):
    return send_from_directory(vdir(vid), name, as_attachment=True)


@app.route("/api/library/<vid>", methods=["DELETE"])
def delete(vid):
    shutil.rmtree(vdir(vid))
    return jsonify(ok=True)


@app.route("/api/archive.zip")
def archive():
    z = Path("/content/archive_all.zip")
    z.unlink(missing_ok=True)
    subprocess.run(["zip", "-r", "-0", "-q", str(z), ROOT.name, "-x", "*/frames/*"],
                   cwd=str(ROOT.parent), check=True)
    return send_file(z, as_attachment=True)


# ---------------- start server + free tunnel ----------------
logging.getLogger("werkzeug").setLevel(logging.ERROR)
if "_server" not in globals():
    _server = threading.Thread(target=lambda: app.run(host="127.0.0.1", port=8000, threaded=True), daemon=True)
    _server.start()


def start_tunnel():
    p = subprocess.Popen([str(CF), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    url, t0 = None, time.time()
    for line in p.stderr:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
        if time.time() - t0 > 90:
            break
    threading.Thread(target=lambda: [None for _ in p.stderr], daemon=True).start()
    return p, url


if "TUNNEL" not in globals() or TUNNEL[0].poll() is not None:
    TUNNEL = start_tunnel()
URL = TUNNEL[1]

from IPython.display import HTML, display
if not URL:
    print("The tunnel did not start. Run this cell again.")
else:
    code = f"{URL}|{KEY}"
    display(HTML(
        '<div style="font:15px sans-serif;padding:16px;border:2px solid #b8860b;border-radius:10px;'
        'background:#fffdf8;color:#1a2a3a;line-height:1.6">'
        '<b style="font-size:17px">Engine is running.</b><br>'
        f'Go back to the YT Archive page and paste this code:<br>'
        f'<code id="c" style="user-select:all;background:#eef3ff;padding:4px 8px;border-radius:5px;'
        f'word-break:break-all">{code}</code> '
        f'<button onclick="navigator.clipboard.writeText(\'{code}\')" style="margin-left:6px;padding:4px 12px;'
        'border:none;border-radius:6px;background:#b8860b;color:#fff;font-weight:bold;cursor:pointer">Copy</button>'
        f'<br><a href="{PAGE_URL}#u={URL}&k={KEY}" target="_blank" style="color:#3a5a82">'
        'Or open the page already connected</a><br>'
        '<small>Leave this cell running. Stopping it ends the session.</small></div>'))
    while True:
        time.sleep(30)
